# Stroke Risk Prediction

**Tabular Classification Project**

## 2 · Project Overview

Predict whether a patient will experience a **stroke** based on demographics, health conditions, and lifestyle. Synthetic dataset with ~5,000 rows and severe class imbalance (~5% positive).

## 3 · Learning Objectives

1. Perform EDA and target analysis on a classification dataset.
2. Handle categorical encoding, missing values, and class imbalance.
3. Build a baseline model and compare against modern boosting models.
4. Use LazyPredict for rapid classifier benchmarking.
5. Run FLAML AutoML for automated model selection.
6. Train CatBoost, LightGBM, and XGBoost with GPU auto-detection.
7. Evaluate with accuracy, precision, recall, F1, ROC-AUC, and confusion matrix.

## 4 · Problem Statement

Given patient demographics (age, gender), health history (hypertension, heart disease, glucose), and lifestyle (smoking, BMI), predict stroke risk.

## 5 · Why This Project Matters

- Stroke is the **#2 cause of death** and #3 cause of disability globally.
- Severe class imbalance (~5% stroke) makes this a challenging ML problem.
- This teaches how to handle **extreme imbalance** — accuracy is misleading.
- Risk scoring enables preventive interventions (medication, lifestyle changes).

## 6 · Dataset Overview

| Property | Value |
|----------|-------|
| Rows | ~5,000 |
| Features | 10 (age, gender, hypertension, heart_disease, ever_married, work_type, residence_type, avg_glucose_level, bmi, smoking_status) |
| Target | `stroke` (binary: 0/1) |
| Class balance | ~95% no stroke, ~5% stroke |
| Missing values | Some in BMI |

## 7 · Dataset Source and License Notes

**Source**: Kaggle Stroke Prediction Dataset (synthetic/educational).
**License**: Public / educational.
**Note**: Synthetic dataset for ML practice.

## 8 · Environment Setup

In [1]:
import subprocess, sys

def _install(pkg):
    try:
        __import__(pkg.replace('-','_'))
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

for pkg in ["catboost", "lightgbm", "xgboost", "flaml", "lazypredict"]:
    _install(pkg)
print("All packages ready.")

C:\Users\ahmad\AppData\Local\Programs\Python\Python313\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.0.post2)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


All packages ready.


## 9 · Imports

In [2]:
import os, json, time, warnings, random
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, confusion_matrix,
                              classification_report, ConfusionMatrixDisplay)

warnings.filterwarnings("ignore")
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
print("Imports complete.")

Imports complete.


## 10 · Configuration / Constants

In [3]:
TARGET = "stroke"
SEED = 42
SAVE_DIR = os.path.dirname(os.path.abspath('__dummy__'))
print(f"Save dir: {SAVE_DIR}")

Save dir: E:\Github\Machine-Learning-Projects\Classification\Stroke Risk Prediction


## 11 · Dataset Download or Loading

In [4]:
np.random.seed(SEED)
n = 5000

age = np.random.beta(2, 1.5, n) * 80 + 1
age = age.clip(1, 82).round(0).astype(int)
gender = np.random.choice(['Male', 'Female'], n)
hypertension = np.random.binomial(1, 0.1 + 0.003 * age)
heart_disease = np.random.binomial(1, 0.03 + 0.002 * age)
ever_married = np.where(age > 25, np.random.choice(['Yes', 'No'], n, p=[0.8, 0.2]),
                        np.random.choice(['Yes', 'No'], n, p=[0.15, 0.85]))
work_type = np.random.choice(['Private', 'Self-employed', 'Govt_job', 'children', 'Never_worked'], n,
                              p=[0.55, 0.15, 0.15, 0.1, 0.05])
residence = np.random.choice(['Urban', 'Rural'], n)
glucose = np.random.lognormal(4.5, 0.4, n).clip(50, 280).round(1)
bmi = np.random.normal(28, 7, n).clip(12, 60).round(1)
smoking = np.random.choice(['never smoked', 'formerly smoked', 'smokes', 'Unknown'], n,
                            p=[0.35, 0.2, 0.15, 0.3])

score = (
    0.04 * (age - 40)
    + 0.8 * hypertension
    + 0.6 * heart_disease
    + 0.01 * (glucose - 100)
    + 0.02 * (bmi - 25)
    + 0.3 * (smoking == 'smokes').astype(float)
    + np.random.normal(0, 1.5, n)
)
stroke = (score > 3.5).astype(int)

df = pd.DataFrame({
    'age': age, 'gender': gender, 'hypertension': hypertension,
    'heart_disease': heart_disease, 'ever_married': ever_married,
    'work_type': work_type, 'Residence_type': residence,
    'avg_glucose_level': glucose, 'bmi': bmi,
    'smoking_status': smoking, 'stroke': stroke,
})
print(f"Dataset shape: {df.shape}")
print(f"Stroke rate: {df['stroke'].mean():.2%}")
df.head()

Dataset shape: (5000, 11)
Stroke rate: 5.34%


,age,gender,hypertension,heart_disease,ever_married,work_type,Residence_type,avg_glucose_level,bmi,smoking_status,stroke
0,57,Male,1,0,Yes,Private,Rural,62.8,33.6,Unknown,0
1,49,Male,0,0,Yes,Self-employed,Urban,50.0,27.4,Unknown,0
2,55,Male,0,0,Yes,Private,Urban,50.0,28.3,Unknown,0
3,31,Female,0,0,Yes,Private,Urban,90.5,32.4,never smoked,0
4,78,Male,0,0,Yes,Self-employed,Rural,137.1,26.3,smokes,0


## 12 · Data Validation Checks

In [5]:
print("=" * 50)
print("DATA VALIDATION")
print("=" * 50)
print(f"\nShape: {df.shape}")
print(f"\nMissing values:\n{df.isnull().sum()[df.isnull().sum() > 0]}")
if df.isnull().sum().sum() == 0:
    print("No missing values.")
print(f"\nDuplicate rows: {df.duplicated().sum()}")
print(f"\nTarget distribution:\n{df[TARGET].value_counts()}")
print(f"\nDtypes:\n{df.dtypes}")
assert TARGET in df.columns, f"Target '{TARGET}' not found!"
print(f"\nTarget '{TARGET}' confirmed.")

DATA VALIDATION

Shape: (5000, 11)

Missing values:
Series([], dtype: int64)
No missing values.

Duplicate rows: 0

Target distribution:
stroke
0    4733
1     267
Name: count, dtype: int64

Dtypes:
age                    int64
gender                object
hypertension           int32
heart_disease          int32
ever_married          object
work_type             object
Residence_type        object
avg_glucose_level    float64
bmi                  float64
smoking_status        object
stroke                 int64
dtype: object

Target 'stroke' confirmed.


## 13 · Exploratory Data Analysis

In [6]:
num_cols = df.select_dtypes(include='number').columns.drop(TARGET).tolist()
cat_cols = df.select_dtypes(include='object').columns.tolist()

fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for i, col in enumerate(num_cols[:4]):
    df.boxplot(column=col, by=TARGET, ax=axes[i])
    axes[i].set_title(f"{col} by Stroke")
plt.suptitle("")
plt.tight_layout()
plt.savefig('eda_numeric.png', dpi=100, bbox_inches='tight')
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for i, col in enumerate(['hypertension', 'heart_disease']):
    ct = pd.crosstab(df[col], df[TARGET], normalize='index')
    ct.plot(kind='bar', stacked=True, ax=axes[i], color=['steelblue', 'coral'])
    axes[i].set_title(f"Stroke Rate by {col}")
plt.tight_layout()
plt.show()
print(f"Numeric: {len(num_cols)}, Categorical: {len(cat_cols)}")

Numeric: 5, Categorical: 5


## 14 · Target Analysis

In [7]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
df[TARGET].value_counts().plot(kind='bar', ax=axes[0], color=['steelblue', 'coral'], edgecolor='black')
axes[0].set_title("Stroke Distribution")
axes[0].set_xticklabels(['No Stroke (0)', 'Stroke (1)'], rotation=0)
df[TARGET].value_counts(normalize=True).plot(kind='pie', ax=axes[1], autopct='%1.1f%%', colors=['steelblue', 'coral'])
axes[1].set_title("Class Proportions"); axes[1].set_ylabel('')
plt.tight_layout(); plt.show()
print(f"Class distribution:\n{df[TARGET].value_counts()}")
print(f"\nImbalance ratio: {df[TARGET].value_counts().iloc[0] / max(df[TARGET].value_counts().iloc[1], 1):.1f}:1")

Class distribution:
stroke
0    4733
1     267
Name: count, dtype: int64

Imbalance ratio: 17.7:1


## 15 · Train/Validation/Test Split Strategy

Stratified 80/20 split to preserve class distribution.

In [8]:
cat_cols = df.select_dtypes(include='object').columns.tolist()
if cat_cols:
    oe = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
    df[cat_cols] = oe.fit_transform(df[cat_cols])

X = df.drop(columns=[TARGET])
y = df[TARGET]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED, stratify=y)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Train target dist: {y_train.value_counts().to_dict()}")

Train: (4000, 10), Test: (1000, 10)
Train target dist: {0: 3786, 1: 214}


## 16 · Preprocessing Strategy

Categorical features encoded via OrdinalEncoder. Missing values handled before split. Tree-based models handle raw features without scaling.

## 17 · Feature Engineering

In [9]:
clean = [c.replace('-', '_').replace(' ', '_').replace('.', '_') for c in X_train.columns]
X_train.columns = clean
X_test.columns = clean
print(f"Features ({len(clean)}): {clean}")

Features (10): ['age', 'gender', 'hypertension', 'heart_disease', 'ever_married', 'work_type', 'Residence_type', 'avg_glucose_level', 'bmi', 'smoking_status']


## 18 · Baseline Model

Logistic Regression as baseline.

In [10]:
baseline = LogisticRegression(max_iter=1000, random_state=SEED)
baseline.fit(X_train, y_train)
y_pred_bl = baseline.predict(X_test)
y_prob_bl = baseline.predict_proba(X_test)[:, 1] if len(np.unique(y_train)) == 2 else None

print("=" * 50)
print("BASELINE: Logistic Regression")
print("=" * 50)
print(f"Accuracy : {accuracy_score(y_test, y_pred_bl):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_bl, average='weighted'):.4f}")
print(f"Recall   : {recall_score(y_test, y_pred_bl, average='weighted'):.4f}")
print(f"F1       : {f1_score(y_test, y_pred_bl, average='weighted'):.4f}")
if y_prob_bl is not None:
    print(f"ROC-AUC  : {roc_auc_score(y_test, y_prob_bl):.4f}")

BASELINE: Logistic Regression
Accuracy : 0.9490
Precision: 0.9516
Recall   : 0.9490
F1       : 0.9260
ROC-AUC  : 0.8274


## 19 · LazyPredict Benchmark

In [11]:
from lazypredict.Supervised import LazyClassifier

lazy = LazyClassifier(verbose=0, ignore_warnings=True)
lazy_models, _ = lazy.fit(X_train, X_test, y_train, y_test)
print("LazyPredict — Top 15 Classifiers:")
print(lazy_models.head(15).to_string())

LazyPredict — Top 15 Classifiers:
                               Accuracy  Balanced Accuracy   ROC AUC  F1 Score  Precision  Recall  Time Taken
Model                                                                                                        
NearestCentroid                   0.757           0.764828  0.826084  0.822987   0.939379   0.757    0.020590
Perceptron                        0.850           0.698153  0.802335  0.882227   0.928748   0.850    0.021768
LabelSpreading                    0.912           0.579486  0.687773  0.913507   0.915050   0.912    0.471602
LabelPropagation                  0.911           0.578958  0.664621  0.912887   0.914829   0.911    0.363399
DecisionTreeClassifier            0.902           0.574207  0.574207  0.907352   0.913071   0.902    0.026814
GaussianNB                        0.939           0.549212  0.820227  0.925735   0.917483   0.939    0.016780
ExtraTreeClassifier               0.900           0.537527  0.537527  0.903297   0.906

## 20 · FLAML AutoML Run

In [12]:
from flaml import AutoML

flaml_automl = AutoML()
flaml_automl.fit(
    X_train, y_train,
    task="classification", time_budget=60, metric="f1",
    estimator_list=['lgbm', 'rf', 'extra_tree', 'catboost'],
    seed=SEED, verbose=0
)
y_pred_flaml = flaml_automl.predict(X_test)
print(f"FLAML best: {flaml_automl.best_estimator}")
print(f"Accuracy : {accuracy_score(y_test, y_pred_flaml):.4f}")
print(f"F1       : {f1_score(y_test, y_pred_flaml, average='weighted'):.4f}")

FLAML best: lgbm
Accuracy : 0.9400
F1       : 0.9251


## 21 · CatBoost, LightGBM, XGBoost

GPU auto-detected with CPU fallback.

In [13]:
import gc

def gpu_cleanup():
    gc.collect()
    try:
        import torch; torch.cuda.empty_cache()
    except Exception:
        pass

results = {}

# CatBoost
from catboost import CatBoostClassifier
try:
    cb = CatBoostClassifier(iterations=500, learning_rate=0.05, depth=6,
                            task_type="GPU", devices="0", verbose=0, random_seed=SEED)
    cb.fit(X_train, y_train)
except Exception:
    cb = CatBoostClassifier(iterations=500, learning_rate=0.05, depth=6,
                            verbose=0, random_seed=SEED)
    cb.fit(X_train, y_train)
results["CatBoost"] = cb.predict(X_test)
print(f"CatBoost  Acc: {accuracy_score(y_test, results['CatBoost']):.4f}  F1: {f1_score(y_test, results['CatBoost'], average='weighted'):.4f}")
gpu_cleanup()

# LightGBM
import lightgbm as lgbm
try:
    lg = lgbm.LGBMClassifier(n_estimators=500, learning_rate=0.05, max_depth=6,
                              device="gpu", verbose=-1, random_state=SEED)
    lg.fit(X_train, y_train)
except Exception:
    lg = lgbm.LGBMClassifier(n_estimators=500, learning_rate=0.05, max_depth=6,
                              verbose=-1, random_state=SEED)
    lg.fit(X_train, y_train)
results["LightGBM"] = lg.predict(X_test)
print(f"LightGBM  Acc: {accuracy_score(y_test, results['LightGBM']):.4f}  F1: {f1_score(y_test, results['LightGBM'], average='weighted'):.4f}")
gpu_cleanup()

# XGBoost
from xgboost import XGBClassifier
try:
    xgb_model = XGBClassifier(n_estimators=500, learning_rate=0.05, max_depth=6,
                               device="cuda", tree_method="hist", verbosity=0,
                               random_state=SEED, use_label_encoder=False, eval_metric="logloss")
    xgb_model.fit(X_train, y_train)
except Exception:
    xgb_model = XGBClassifier(n_estimators=500, learning_rate=0.05, max_depth=6,
                               tree_method="hist", verbosity=0,
                               random_state=SEED, use_label_encoder=False, eval_metric="logloss")
    xgb_model.fit(X_train, y_train)
results["XGBoost"] = xgb_model.predict(X_test)
print(f"XGBoost   Acc: {accuracy_score(y_test, results['XGBoost']):.4f}  F1: {f1_score(y_test, results['XGBoost'], average='weighted'):.4f}")
gpu_cleanup()

results["Logistic Regression"] = y_pred_bl
results["FLAML"] = y_pred_flaml

CatBoost  Acc: 0.9450  F1: 0.9238


LightGBM  Acc: 0.9420  F1: 0.9250


XGBoost   Acc: 0.9420  F1: 0.9264


## 22 · Top 3 Model Selection

In [14]:
model_scores = {}
for name, yp in results.items():
    model_scores[name] = {
        "Accuracy": round(accuracy_score(y_test, yp), 4),
        "F1": round(f1_score(y_test, yp, average='weighted'), 4),
        "Precision": round(precision_score(y_test, yp, average='weighted'), 4),
        "Recall": round(recall_score(y_test, yp, average='weighted'), 4),
    }

scores_df = pd.DataFrame(model_scores).T.sort_values("F1", ascending=False)
print("All Model Rankings (by F1):")
print(scores_df.to_string())
top3_names = scores_df.index[:3].tolist()
print(f"\nTop 3: {top3_names}")

All Model Rankings (by F1):
                     Accuracy      F1  Precision  Recall
XGBoost                 0.942  0.9264     0.9185   0.942
Logistic Regression     0.949  0.9260     0.9516   0.949
FLAML                   0.940  0.9251     0.9163   0.940
LightGBM                0.942  0.9250     0.9163   0.942
CatBoost                0.945  0.9238     0.9161   0.945

Top 3: ['XGBoost', 'Logistic Regression', 'FLAML']


## 23 · Final Training and Evaluation of Top 3

In [15]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for i, name in enumerate(top3_names):
    yp = results[name]
    cm = confusion_matrix(y_test, yp)
    ConfusionMatrixDisplay(cm).plot(ax=axes[i], cmap='Blues')
    f1 = f1_score(y_test, yp, average='weighted')
    acc = accuracy_score(y_test, yp)
    axes[i].set_title(f"{name}\nAcc={acc:.4f} F1={f1:.4f}")
plt.suptitle("Top 3 — Confusion Matrices", fontsize=14)
plt.tight_layout()
plt.savefig('top3_confusion.png', dpi=100, bbox_inches='tight')
plt.show()

print("\nDetailed Classification Reports — Top 3:")
for name in top3_names:
    print(f"\n{'='*50}")
    print(f"  {name}")
    print('='*50)
    print(classification_report(y_test, results[name]))


Detailed Classification Reports — Top 3:

  XGBoost
              precision    recall  f1-score   support

           0       0.95      0.99      0.97       947
           1       0.33      0.09      0.15        53

    accuracy                           0.94      1000
   macro avg       0.64      0.54      0.56      1000
weighted avg       0.92      0.94      0.93      1000


  Logistic Regression
              precision    recall  f1-score   support

           0       0.95      1.00      0.97       947
           1       1.00      0.04      0.07        53

    accuracy                           0.95      1000
   macro avg       0.97      0.52      0.52      1000
weighted avg       0.95      0.95      0.93      1000


  FLAML
              precision    recall  f1-score   support

           0       0.95      0.99      0.97       947
           1       0.29      0.09      0.14        53

    accuracy                           0.94      1000
   macro avg       0.62      0.54      0.56

## 24 · Error Analysis

In [16]:
best_name = top3_names[0]
best_pred = results[best_name]
errors = y_test != best_pred
error_rate = errors.mean()
print(f"Best model: {best_name}")
print(f"Error rate: {error_rate:.4f} ({errors.sum()} / {len(y_test)})")
print(f"\nErrors by true class:")
error_df = pd.DataFrame({'true': y_test, 'pred': best_pred, 'error': errors})
print(error_df.groupby('true')['error'].agg(['sum', 'count', 'mean']).rename(
    columns={'sum': 'errors', 'count': 'total', 'mean': 'error_rate'}))

Best model: XGBoost
Error rate: 0.0580 (58 / 1000)

Errors by true class:
      errors  total  error_rate
true                           
0         10    947     0.01056
1         48     53     0.90566


## 25 · Interpretation and Business Insight

- **Age** is the dominant stroke risk factor — risk increases sharply after 60.
- **Hypertension** and **heart disease** are strong clinical predictors.
- **Glucose levels** above 200 indicate significantly higher risk.
- **Smoking** contributes but is often encoded as 'Unknown'.
- Extreme class imbalance means most models predict 'no stroke' for everyone.

## 26 · Limitations

1. Severe class imbalance (~5% stroke) makes evaluation tricky.
2. Synthetic data — real stroke risk depends on many more factors.
3. 'Unknown' smoking status is common and informative.
4. No temporal data (stroke timing, symptom progression).
5. Missing important risk factors (cholesterol, family history, medications).

## 27 · How to Improve This Project

1. Apply SMOTE or class weights for better recall on strokes.
2. Use PR-AUC instead of ROC-AUC for imbalanced evaluation.
3. Engineer age groups and glucose risk categories.
4. Add clinical risk scores (CHA₂DS₂-VASc) as features.
5. Threshold tuning to optimize recall at acceptable precision.

## 28 · Production Considerations

- Stroke risk screening in primary care settings.
- Integration with clinical risk calculators.
- Alerting system for high-risk patients.
- Must be validated on local population before deployment.

## 29 · Common Mistakes

1. Using accuracy on 95/5 split (95% baseline!).
2. Not using class weights or resampling.
3. Treating 'Unknown' smoking as missing instead of a category.
4. Not evaluating recall specifically on stroke cases.
5. Ignoring age as a dominant confounding factor.

## 30 · Mini Challenge / Exercises

1. Calculate accuracy of a 'predict no stroke always' dummy — what's the baseline?
2. Apply class_weight='balanced' to LogisticRegression — how does recall change?
3. Plot precision-recall curve instead of ROC curve.
4. Build an age-only model — how close to the full model?

## 31 · Final Summary / Key Takeaways

1. Stroke prediction showcases extreme class imbalance challenges.
2. Age, hypertension, and heart disease dominate risk.
3. Accuracy is misleading — use F1, recall, PR-AUC.
4. Class weights or resampling are essential for minority-class recall.
5. Clinical deployment requires population-specific calibration.

## Save Metrics

In [17]:
metrics_out = {}
for name, yp in results.items():
    metrics_out[name] = {
        "accuracy": round(float(accuracy_score(y_test, yp)), 4),
        "f1": round(float(f1_score(y_test, yp, average='weighted')), 4),
        "precision": round(float(precision_score(y_test, yp, average='weighted')), 4),
        "recall": round(float(recall_score(y_test, yp, average='weighted')), 4),
    }
with open('metrics.json', 'w') as f:
    json.dump(metrics_out, f, indent=2)
print("Metrics saved.")
print(json.dumps(metrics_out, indent=2))

Metrics saved.
{
  "CatBoost": {
    "accuracy": 0.945,
    "f1": 0.9238,
    "precision": 0.9161,
    "recall": 0.945
  },
  "LightGBM": {
    "accuracy": 0.942,
    "f1": 0.925,
    "precision": 0.9163,
    "recall": 0.942
  },
  "XGBoost": {
    "accuracy": 0.942,
    "f1": 0.9264,
    "precision": 0.9185,
    "recall": 0.942
  },
  "Logistic Regression": {
    "accuracy": 0.949,
    "f1": 0.926,
    "precision": 0.9516,
    "recall": 0.949
  },
  "FLAML": {
    "accuracy": 0.94,
    "f1": 0.9251,
    "precision": 0.9163,
    "recall": 0.94
  }
}
